# 00 · Start here: what RAG is, from scratch

**You'll learn:** the problem this project solves, the idea behind retrieval-augmented generation (RAG), and the five building blocks (tokens, embeddings, keyword search, rank fusion, grounded prompts), each with a tiny hands-on demo.

**Time:** about 15 minutes · **Cost:** under 1¢ (a few embedding calls and one short LLM call) · **Needs:** an API key in `.env`. No database needed for this notebook.

| Part | What it covers |
|---|---|
| 1 | The problem |
| 2 | The idea: retrieval-augmented generation |
| 3 | Check your setup |
| 4 | The building blocks, hands-on: 4.1 tokens · 4.2 embeddings · 4.3 keyword search (BM25) · 4.4 rank fusion (RRF) · 4.5 grounded prompts and citations |
| 5 | Where to go next |

Run the cells top to bottom with **Shift + Enter**.

---
## Part 1 · The problem

A company's annual report to the US regulator (the **10-K**) is long and dense. Apple's is 121 pages of text, risk factors and financial tables. Say you want one fact: *"Why did Greater China sales fall in 2024?"*

| Approach | What goes wrong |
|---|---|
| Read it yourself | Slow. The answer is one paragraph somewhere in Item 7. |
| Ask a chatbot directly | It answers from memory: maybe outdated, maybe invented, and with no source you can check. |
| Paste the whole PDF into the chatbot | Expensive (tens of thousands of tokens every question), may not fit, and still hard to verify. |

What we want: an answer that comes **only** from the filing, **cites** the exact passages, and **says "not found"** when the filing doesn't cover the question.

---
## Part 2 · The idea: retrieval-augmented generation

RAG splits the work in two:

1. **Retrieve**: find the handful of passages in the filing that are relevant to the question.
2. **Generate**: give only those passages to an LLM, with strict instructions to answer from them and cite them.

The project runs this in two phases:

```mermaid
flowchart LR
  subgraph P1["Phase 1 · Ingest (run once)"]
    direction LR
    A["10-K PDF"] --> B["Chunk into passages"] --> C["Embed: dense + sparse vectors"] --> D[("Qdrant vector DB")]
  end
  subgraph P2["Phase 2 · Answer (every question)"]
    direction LR
    Q["Question"] --> S["Hybrid search"] --> K["Top 8 passages"] --> G["Grounded prompt + LLM"] --> R["Answer + citations, or 'not found'"]
  end
  D -. searched by .-> S
```

Phase 1 is notebook **01**, Phase 2 is notebook **02**. Before that, Part 4 below shows each building block on toy examples.

---
## Part 3 · Check your setup

### Setup
The next cell connects the notebook to the project. Read its first lines: `USE_EMBEDDED_QDRANT` picks where the database runs (Docker server, or embedded with no Docker). It prints which LLM, embedding model and database this run uses, all taken from your `.env`.

In [ ]:
# ── Settings ────────────────────────────────────────────────────────────
# False: use the Qdrant server from `docker compose up -d`.
# True:  no Docker. Qdrant runs inside this notebook and stores data in data/qdrant_local/.
#        Restart the kernel after changing this, and run one notebook at a time
#        (only one process can open the embedded folder).
USE_EMBEDDED_QDRANT = False

# Setup: run from the repo root so `core`/`ingestion` imports and data/ paths work.
import logging
import os
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
logging.basicConfig(level=logging.WARNING)
if USE_EMBEDDED_QDRANT:
    os.environ["QDRANT_PATH"] = "data/qdrant_local"
warnings.filterwarnings("ignore", message="Qdrant client version")  # harmless client/server version notice

import pandas as pd

import core  # noqa: F401  (loads .env)
import core.embeddings as emb
import core.llm as llm
from core.clients import COLLECTION_NAME, QDRANT_PATH, QDRANT_URL, get_qdrant_client

pd.set_option("display.max_colwidth", 120)
print(f"LLM:        {llm.PROVIDER} / {llm.MODEL}")
print(f"Embeddings: {emb.PROVIDER} / {emb.MODEL}")
print(f"Qdrant:     {'embedded @ ' + QDRANT_PATH if QDRANT_PATH else 'server @ ' + QDRANT_URL}")

This cell checks that the key for your chosen provider is present (it prints only yes or no, never the key itself).

In [ ]:
import platform

needed = {llm.PROVIDER, emb.PROVIDER}
keys = {"openai": "OPENAI_API_KEY", "gemini": "GEMINI_API_KEY"}
print("Python:", platform.python_version())
for provider in sorted(needed):
    print(f"{keys[provider]} set:", "yes" if os.environ.get(keys[provider]) else "NO: add it to .env, then restart the kernel")

---
## Part 4 · The building blocks, hands-on

### 4.1 Tokens: how models measure text

Models don't read characters or words; they read **tokens**, pieces of words. Everything is counted in tokens: how much text fits in one request, how much it costs, and how big our chunks are. Rule of thumb: **1 token ≈ ¾ of a word**, so 800 tokens ≈ 600 words.

In [ ]:
sentence = "Greater China net sales decreased during 2024 compared to 2023."

# Option A: OpenAI (tiktoken, local). Shows the individual tokens.
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o")
token_ids = enc.encode(sentence)
print(f"{len(sentence.split())} words -> {len(token_ids)} tokens")
print([enc.decode([t]) for t in token_ids])

# Option B: Gemini (API call). Gemini only returns the count, not the token pieces.
# import os
# from google import genai
#
# client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
# result = client.models.count_tokens(model="gemini-3.5-flash", contents=sentence)
# print(f"{len(sentence.split())} words -> {result.total_tokens} tokens")

# Option C: Gemini tokens shown locally. Uses Google's open Gemma 3 tokenizer, which is
# the one the google-genai SDK uses for its local tokenizer. Needs sentencepiece (in pyproject).
# (an approximation: newer Gemini models may differ slightly).
# import urllib.request
# import sentencepiece as spm
#
# url = "https://raw.githubusercontent.com/google/gemma_pytorch/014acb7ac4563a5f77c76d7ff98f31b568c16508/tokenizer/gemma3_cleaned_262144_v2.spiece.model"
# sp = spm.SentencePieceProcessor(model_proto=urllib.request.urlopen(url).read())
# pieces = sp.encode(sentence, out_type=str)
# print(f"{len(sentence.split())} words -> {len(pieces)} tokens")
# print(pieces)  # "▁" marks a leading space

**What to notice:** common words are one token; rarer pieces split up. This is why the project measures chunk size in tokens (the advanced chunker caps chunks at 800).

### 4.2 Embeddings: turning meaning into numbers

An **embedding** is a list of numbers (here 1536 of them) that represents what a text *means*. Texts with similar meaning get similar lists, even if they share no words. We compare two embeddings with **cosine similarity**: 1.0 = same direction (same meaning), near 0 = unrelated.

The cell embeds five sentences plus a question, and ranks the sentences by similarity to the question.

In [ ]:
import numpy as np

from core.embeddings import embed_dense

sentences = [
    "Total net sales were $391.0 billion in 2024.",
    "Apple's revenue increased compared to the prior year.",
    "iPhone sales in Greater China declined.",
    "The company faces risks from foreign currency fluctuations.",
    "The weather in Paris is sunny today.",
]
question = "How much money did Apple make from sales?"

vectors = np.array(embed_dense(sentences + [question]))
vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)  # unit length, so dot product = cosine
similarity = vectors[:-1] @ vectors[-1]

print(f"Each embedding has {vectors.shape[1]} numbers.\n")
pd.DataFrame({"sentence": sentences, "cosine similarity to the question": similarity.round(3)}).sort_values(
    "cosine similarity to the question", ascending=False
)

**What to notice:** the sentences about sales and revenue rank highest even though the question says "money" and "make", not "net sales". The Paris sentence ranks last. That is **dense (semantic) search**: it matches meaning.

### 4.3 Keyword search (BM25): matching exact words

Dense search can miss exact names, numbers and jargon such as "Item 1C" or "Greater China". **BM25** is classic keyword scoring: a text scores higher when it contains the query's words, especially rare ones. In Qdrant each text becomes a **sparse vector** (a weight for each word it contains), and the score is the dot product of the query's and the text's sparse vectors.

In [ ]:
from core.clients import get_sparse_model


def sparse_score(a, b):
    weights = dict(zip(a.indices.tolist(), a.values.tolist()))
    return sum(weights.get(i, 0.0) * v for i, v in zip(b.indices.tolist(), b.values.tolist()))


bm25 = get_sparse_model()
keyword_query = "Greater China net sales"
doc_vectors = list(bm25.embed(sentences))
query_vector = next(bm25.query_embed(keyword_query))

pd.DataFrame({
    "sentence": sentences,
    "BM25 score": [round(sparse_score(query_vector, d), 3) for d in doc_vectors],
}).sort_values("BM25 score", ascending=False)

**What to notice:** only sentences sharing words with the query score above 0; "revenue increased" scores 0 even though it means something close. Dense and sparse search fail in opposite ways, so the project uses **both** ("hybrid search").

### 4.4 Rank fusion (RRF): combining two result lists

Hybrid search gives two ranked lists. Their scores aren't comparable (cosine is 0 to 1; BM25 is unbounded), so we combine **ranks** instead. **Reciprocal Rank Fusion** gives each passage `1 / (k + rank)` from each list and adds them up. `k` (60 is the classic value) softens the gap between rank 1 and rank 2. A passage near the top of *both* lists wins.

In [ ]:
dense_ranking = ["Greater China paragraph", "Segment table", "Risk: China", "Revenue overview"]
sparse_ranking = ["Segment table", "Greater China paragraph", "Store count in China", "Risk: China"]
K = 60

scores = {}
for ranking in (dense_ranking, sparse_ranking):
    for rank, passage in enumerate(ranking, start=1):
        scores[passage] = scores.get(passage, 0) + 1 / (K + rank)

pd.DataFrame(
    [
        {
            "passage": p,
            "dense rank": dense_ranking.index(p) + 1 if p in dense_ranking else "-",
            "sparse rank": sparse_ranking.index(p) + 1 if p in sparse_ranking else "-",
            "RRF score": round(s, 4),
        }
        for p, s in sorted(scores.items(), key=lambda kv: -kv[1])
    ]
)

**What to notice:** passages found by both searches rise to the top; one found by a single search drops. The real pipeline keeps the **top k = 8** after fusion (notebook 02, Part 2).

### 4.5 Grounded prompts and citations

The last block is the prompt. The retrieved passages are numbered, and the LLM is told: *answer using ONLY these excerpts, then write `CITED:` with the numbers you used*. The code then turns those numbers back into citations. No citations means the app answers "not found".

Here are two made-up excerpts run through the project's real prompt builder, an LLM call, and the citation parser:

In [ ]:
from core.llm import complete_with_usage
from core.models import RetrievedChunk
from core.rag import _build_prompt, _parse_response

toy_chunks = [
    RetrievedChunk(id=0, ticker="AAPL", company_name="Apple Inc.", filing_type="10-K", section="Item 7",
                   fiscal_year="2024", text="Greater China net sales decreased due to lower iPhone sales.", score=1.0),
    RetrievedChunk(id=1, ticker="AAPL", company_name="Apple Inc.", filing_type="10-K", section="Item 1A",
                   fiscal_year="2024", text="The Company is exposed to foreign currency risk.", score=0.5),
]
prompt = _build_prompt("Why did Greater China sales fall?", toy_chunks)
print(prompt, "\n\n----- LLM reply -----")
reply = complete_with_usage(prompt, max_tokens=200)
print(reply.text)
answer, citations = _parse_response(reply.text, toy_chunks)
print("\nParsed citations:", [c.section for c in citations])

**What to notice:** the reply ends with a `CITED:` line pointing at excerpt `0`, and the parser turned it into a citation for Item 7. Ask something the excerpts don't cover and `CITED:` comes back empty, which is how the app knows to say "not found".

---
## Part 5 · Where to go next

| Notebook | What you'll do |
|---|---|
| **01_ingest** | Phase 1 for real: fetch the 10-K, chunk it two ways, embed, and index into Qdrant |
| **02_answer** | Phase 2 for real: follow one question through search, fusion, prompt, LLM and citations |
| **03_models_and_experiments** | Compare OpenAI vs Gemini, the two chunkers, five retrieval strategies, and run a mini evaluation |
| **04_agentic_rag** | Turn this pipeline into an agent that decides what to search for, and how often |

Each notebook runs on its own: it builds any missing data itself.